# Module 8.1: Sparse Mixture of Experts (MoE)

Welcome to the bleeding edge of Transformer Architecture!

By now, you have built the entire "Dense" Transformer model. Every single token inputted into the model is multiplied against every single mathematical weight in the network. 

In this notebook, we look at the ultimate architectural trick designed to scale models to trillions of parameters *without* destroying latency speeds: **Mixture of Experts**.

## 1. WHAT is Mixture of Experts (MoE)?

In a standard Transformer, the Feed-Forward Network (FFN) located inside every Encoder/Decoder block is a massive block of parameters. 

**MoE** breaks this single massive FFN into multiple smaller FFNs called "Experts". Instead of a token passing through one giant matrix, a special "Router" network looks at the token and sends it to only the top $K$ experts (usually 2)! 

### \ud83c\udfd7\ufe0f The Construction Analogy
Imagine you are building a house (Predicting a token).
- **Dense Network**: You have one "Jack-of-all-Trades" worker. Every time a task arises (Plumbing, Electricity, Painting), this one poor guy has to do it. If the house gets complex, you have to replace him with an incredibly expensive, slow super-worker.
 
- **MoE**: You hire 8 specialized workers (The Experts) and 1 Foreman (The Router). When a "pipe" task arrives, the Foreman instantly points to the Plumber. The other 7 workers sit idle. It is immensely fast, and strictly specialized!

## 2. WHY do we use MoE?

**Decoupling parameter count from compute.**

If you want a smarter model, you usually make it wider and deeper. But a model with 8x the parameters in its feed-forward layers costs roughly 8x the compute per token.

With an MoE like *Mixtral 8x7B*, the model holds roughly 47 billion parameters in memory (VRAM). But because the router only activates 2 of the 8 experts per token, each token passes through about 13 billion parameters of math. The result is roughly the quality of a much larger dense model at the inference speed of a far smaller one. This is called **sparse scaling**.

(The figures are illustrative of the trade-off, not a precise benchmark of any one model.)

## 3. HOW does MoE work? (The Architecture)

The attention mechanism works exactly as before. The change happens at the feed-forward layer.

1. The input token hits the **router network** (a simple linear layer turned into probabilities).
2. The router scores how much it "likes" each of the 8 experts.
3. We pick the `top-k` scores (e.g. top 2).
4. The token is routed to those two expert FFNs.
5. Their outputs are added together, weighted by the router's confidence scores.

```mermaid
graph TD
    Token[Input Token<br>Dim: 4096] --> Router[Router Network<br>Outputs 8 Scores]

    Router -->|Top 1: Score 0.7| E1[Expert 1<br>Feed-Forward]
    Router -.->|Not Chosen: Score 0.05| E2[Expert 2]
    Router -.->|Not Chosen: Score 0.01| E3[Expert 3]
    Router -->|Top 2: Score 0.2| E4[Expert 4<br>Feed-Forward]

    E1 -->|Multiply by 0.7| Sum{+}
    E4 -->|Multiply by 0.2| Sum

    Sum --> Out[Output Prediction]
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

class Expert(nn.Module):
    """
    A standard Feed-Forward Network. Same as we built in Module 3!
    """
    def __init__(self, d_model):
        super().__init__()
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model)
        )
    def forward(self, x):
        return self.ffn(x)

class SparseMixtureOfExperts(nn.Module):
    def __init__(self, d_model=128, num_experts=8, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k

        # The Foreman: looks at the token (d_model) and outputs (num_experts) scores
        self.router = nn.Linear(d_model, num_experts, bias=False)

        # The Workers: a list of independent FFNs.
        self.experts = nn.ModuleList([Expert(d_model) for _ in range(num_experts)])

    def forward(self, x):
        # x shape: (Batch, Sequence, d_model)

        # 1. The Router "grades" the token for all experts -> (Batch, Sequence, num_experts)
        router_logits = self.router(x)
        routing_probs = F.softmax(router_logits, dim=-1)

        # 2. Pick the Top-K experts per token.
        # top_k_probs / top_k_indices shape: (Batch, Sequence, top_k)
        top_k_probs, top_k_indices = torch.topk(routing_probs, self.top_k, dim=-1)

        # Re-normalize the probabilities of just the chosen experts so they sum to 1.
        top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)

        final_output = torch.zeros_like(x)

        # 3. Route the data, one expert at a time.
        # NOTE (efficiency): this educational version runs each selected expert on the FULL
        # input and then masks out the tokens it didn't get. That means we still pay roughly
        # DENSE compute here. The real speedup requires GATHERING each expert's tokens BEFORE
        # the FFN so an expert only ever does math on the tokens routed to it. We keep the
        # masked version because it is much easier to read.
        for i, expert in enumerate(self.experts):
            # Where was expert `i` selected, across any of the top_k slots?
            expert_mask = (top_k_indices == i)  # (Batch, Sequence, top_k), boolean

            if expert_mask.any():
                expert_out = expert(x)  # (Batch, Sequence, d_model)

                # For each of the top_k slots, add this expert's contribution where it was chosen,
                # weighted by the router's (re-normalized) confidence for that slot.
                for k in range(self.top_k):
                    mask_for_this_k = expert_mask[..., k]          # (Batch, Sequence) boolean
                    gate = top_k_probs[..., k].unsqueeze(-1)       # (Batch, Sequence, 1)
                    final_output[mask_for_this_k] += expert_out[mask_for_this_k] * gate[mask_for_this_k]

        return final_output

# Look at the logic in action!
moe_layer = SparseMixtureOfExperts(d_model=16, num_experts=8, top_k=2)
dummy_tokens = torch.randn(1, 4, 16)  # 1 sentence, 4 words.

print("Pushing tokens through the Mixture of Experts!")
out = moe_layer(dummy_tokens)
print(f"Final Output Shape: {out.shape} -> matches the dense input shape.\n")

# --- Sanity check: compare against a simple per-token reference implementation ---
def reference_moe(moe, x):
    """Slow, obvious version: loop over every token and gather its top-k experts by hand."""
    logits = moe.router(x)
    probs = F.softmax(logits, dim=-1)
    tk_probs, tk_idx = torch.topk(probs, moe.top_k, dim=-1)
    tk_probs = tk_probs / tk_probs.sum(dim=-1, keepdim=True)
    B, S, _ = x.shape
    out = torch.zeros_like(x)
    for b in range(B):
        for s in range(S):
            for k in range(moe.top_k):
                e = tk_idx[b, s, k].item()
                out[b, s] += moe.experts[e](x[b, s]) * tk_probs[b, s, k]
    return out

ref = reference_moe(moe_layer, dummy_tokens)
print(f"Matches the per-token reference implementation? {torch.allclose(out, ref, atol=1e-5)}")
print("\nSparse routing works correctly.")

## 4. Load Balancing (keeping the router honest)

There is a failure mode in MoE training. If left alone, the router often learns to favor a few experts and ignore the rest. In the worst case it sends nearly every token to one expert, which "collapses" the model back to something dense and wastes the others entirely.

To prevent this, MoE training adds an **auxiliary load-balancing loss** to the main loss. The idea: encourage the *fraction of tokens* sent to each expert to be close to uniform. A common form (from the Switch Transformer) is:

$$\text{aux loss} = \alpha \cdot N \cdot \sum_{i=1}^{N} f_i \cdot P_i$$

where $N$ is the number of experts, $f_i$ is the fraction of tokens that picked expert $i$, $P_i$ is the average router probability assigned to expert $i$, and $\alpha$ is a small weight. This is minimized when the load is spread evenly. Let's compute it on our router's output.

In [ ]:
def load_balancing_loss(routing_probs, top_k_indices, num_experts, alpha=0.01):
    """
    Switch-Transformer-style auxiliary loss.
    routing_probs:  (Batch, Sequence, num_experts) softmax router probabilities
    top_k_indices:  (Batch, Sequence, top_k) chosen expert ids
    """
    # f_i: fraction of (token, slot) selections that landed on each expert.
    one_hot = F.one_hot(top_k_indices, num_classes=num_experts).float()  # (B, S, top_k, N)
    f = one_hot.sum(dim=(0, 1, 2)) / one_hot.sum()                        # (N,)

    # P_i: average router probability assigned to each expert.
    P = routing_probs.mean(dim=(0, 1))                                    # (N,)

    return alpha * num_experts * torch.sum(f * P)

# Recompute the router outputs for our dummy tokens.
logits = moe_layer.router(dummy_tokens)
probs = F.softmax(logits, dim=-1)
_, idx = torch.topk(probs, moe_layer.top_k, dim=-1)

aux = load_balancing_loss(probs, idx, moe_layer.num_experts)
print(f"Auxiliary load-balancing loss: {aux.item():.4f}")

# A perfectly uniform router would give the lowest possible value of alpha * 1.0 (= 0.01 here).
print(f"Uniform-router lower bound:    {0.01:.4f}")
print("\nDuring training: total_loss = cross_entropy_loss + aux_loss")

## Summary

By adding a router and replacing one dense FFN with a list of experts, MoE adds parameters (capacity) without making every token pay for all of them. A router picks the top-k experts per token, their outputs are combined by confidence, and an auxiliary load-balancing loss keeps the router from collapsing onto a few experts.

Keep in mind the efficiency caveat from the code: our educational version still runs every selected expert on the full input and masks. The real-world speedup comes from gathering each expert's tokens *before* the FFN so an expert only computes on the tokens routed to it. That bookkeeping is what production MoE kernels optimize.

### 🏋️ Try it yourself

1. **Force a collapse.** Manually overwrite the router weights so it always strongly prefers expert 0 (e.g. `with torch.no_grad(): moe_layer.router.weight.zero_(); moe_layer.router.weight[0] += 5.0`). Re-run a batch and recompute `load_balancing_loss`. Confirm it is much higher than for the balanced router.
2. **Vary top_k.** Build MoE layers with `top_k = 1`, `2`, and `4` on the same input and confirm each still matches its own `reference_moe`. How does increasing `top_k` change how dense the computation becomes?

In [ ]:
# Your code here!
# Hint for task 2:
# for k in [1, 2, 4]:
#     m = SparseMixtureOfExperts(d_model=16, num_experts=8, top_k=k)
#     o = m(dummy_tokens)
#     print(k, torch.allclose(o, reference_moe(m, dummy_tokens), atol=1e-5))